# AI for peptide identification

*Data-driven rescoring with ms2rescore, using MS²PIP and DeepLC*

Every peptide-spectrum match (PSM) reported by a database search engine is a guess: the engine picks, from the database, the peptide whose theoretical fragmentation pattern best explains an observed MS2 spectrum. Some of these guesses are correct and some are not. The search engine's own score is often not the best way to tell them apart.

In this tutorial, we improve on that score using machine learning. We will explore the following steps:

1. Load and inspect the raw search engine results.
2. Predict how each peptide should behave in the LC-MS/MS system (retention time, fragmentation), using two machine learning models: DeepLC and MS²PIP.
3. Compare these predictions to what was actually observed. The agreement, or disagreement, becomes a new and independent piece of evidence.
4. Combine this evidence with the original search engine score to rescore the PSMs, separating correct from incorrect matches more effectively.
5. Use the decoy matches (reversed or scrambled peptides that cannot be real) to set a statistically sound confidence threshold, the false discovery rate (FDR).

Percolator pioneered step 4, a classifier trained on the target/decoy labels. It only works, however, on features the search engine itself already reports. MS²Rescore adds steps 2 and 3 in front of that: it generates extra, independent, ML-predicted features (as we do below) before handing everything to a Percolator-style classifier. The same predict-compare-combine idea, applied to DDA data, also powers MSBooster and Oktoberfest. Related ideas underlie parts of DIA analysis tools such as DIA-NN.

In [ ]:
%pip install -q "ms2pip==4.2.*" "spectrum-utils==0.5.*" "pynumpress==0.0.*" "mzpeak @ git+https://github.com/HUPO-PSI/mzPeak.git@96708ffaed85b1f084bad318c68c006fee147206#subdirectory=python"

In [ ]:
import ast
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import ms2pip
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spectrum_utils.plot as sup
import spectrum_utils.spectrum as sus
from mzpeak import MzPeakFile
from plotly.subplots import make_subplots

# Download data files
DATA_DIR = Path("../data/dda-rescoring")
DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    DATA_DIR / "results.sage.ms2rescore.tsv": "1PTZhoMqZcVAI4JvWEvH3mnL9Wtpe2_la",
    DATA_DIR / "LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_01.mzpeak": "1KHhvC64ZAswBVromB7qd_mvNqkvi0q40",
}

for path, file_id in files.items():
    if not path.exists():
        gdown.download(id=file_id, output=str(path), quiet=False)

## 1. Loading and inspecting the search results

We start from the identifications of [Sage](https://github.com/lazear/sage), a fast open-source search engine, after [MS²Rescore](https://ms2rescore.readthedocs.io/) has already processed it. The resulting table carries, for every PSM, the original Sage result, every predicted feature from DeepLC and MS²PIP, and the final rescored result, all in one place. Working from this file lets us explore each stage of the pipeline directly, without re-running the search or the prediction models ourselves.

For every spectrum, Sage searches the real protein database and a decoy database (reversed or scrambled sequences) together, and reports only the single best-scoring match overall. Because targets and decoys compete on equal footing, that best match is sometimes a decoy. 

In this tutorial, we are using a reprocessed LC-MS run from the 2022 LFQ benchmark dataset by [Van Puyvelde et al.](https://doi.org/10.1038/s41597-022-01216-6) ([PXD028735](https://www.ebi.ac.uk/pride/archive/projects/PXD028735)).

In [ ]:
df = pd.read_csv(DATA_DIR / "results.sage.ms2rescore.tsv", sep="\t")
df["protein_list"] = df["protein_list"].apply(ast.literal_eval)
df.head()

### 1.1 Basic statistics

A few simple counts already tell us a lot about the identification depth of this experiment: how many spectra were matched, how many distinct precursors (peptidoform and charge combined) and peptidoforms (sequence and modifications, any charge) that represents, and how many stripped peptides and proteins those map back to.

In [ ]:
peptidoform_no_charge = df["peptidoform"].str.rsplit("/", n=1).str[0]
stripped_peptide = peptidoform_no_charge.str.replace(r"\[.*?\]", "", regex=True)

counts = pd.Series({
    "PSMs (incl. decoys)": len(df),
    "Precursors": df["peptidoform"].nunique(),
    "Peptidoforms": peptidoform_no_charge.nunique(),
    "Stripped peptides": stripped_peptide.nunique(),
    "Proteins": df["protein_list"].explode().nunique(),
})

fig = px.bar(
    x=counts.index, y=counts.values, text=counts.values,
    labels={"x": "", "y": "Count"}, title="Identification depth",
)
fig.show()

Note that these are all matched items, including decoys, and all matches that fall below the confidence threshold. The numbers of confidently identified items will be lower.

In [ ]:
peptide_proteins = (
    df.assign(sequence=stripped_peptide)
    .drop_duplicates("sequence")[["sequence", "protein_list"]]
)
shared = (peptide_proteins["protein_list"].apply(len) > 1).sum()

print(f"Peptides mapping to exactly one protein: {len(peptide_proteins) - shared:>8,}")
print(f"Peptides mapping to more than one protein: {shared:>6,}")

*The number of PSMs is larger than the number of unique peptides. Name two reasons a single peptide can be matched more than once. [1.1a]*

*As the counts above show, a sizeable share of peptides map to more than one protein: each is equally good evidence for any of those proteins, so the search alone cannot say which protein is truly present. What is this phenomenon called, and why does it complicate reporting "identified proteins"? [1.1b]*

### 1.2 Score distributions and the FDR threshold

Sage assigns every PSM a score. If the score were a perfect indicator of correctness, all correct (target) matches would score higher than all incorrect ones. In practice, the two distributions overlap.

Decoy matches are, by construction, always incorrect. Their score distribution therefore estimates how the incorrect part of the target distribution should look, which is the basis of target-decoy FDR estimation.

In [ ]:
fig = px.histogram(
    df, x="provenance:before_rescoring_score", color="is_decoy", barmode="overlay",
    labels={"is_decoy": "Decoy", "provenance:before_rescoring_score": "Sage score"},
    title="Sage score distribution",
)

threshold_score = df.loc[
    (df["provenance:before_rescoring_qvalue"] <= 0.01) & (~df["is_decoy"]),
    "provenance:before_rescoring_score",
].min()
fig.add_vline(x=threshold_score, line_dash="dash", annotation_text="1% FDR")
fig.show()

*Which PSMs are accepted at the 1% FDR threshold: the ones to the left of the dashed line, or the ones to the right? [1.2a]*

### 1.3 Visualizing a spectrum

A score is an abstraction. Let's look at what a PSM actually looks like: the observed MS2 spectrum, with the matched b- and y-ions annotated on top.

We read spectra directly from an [mzPeak](https://github.com/HUPO-PSI/mzPeak) file, a new standardized mass spectrometry file format that is much faster to query than mzML.

In [ ]:
mzpeak_path = DATA_DIR / "LFQ_Orbitrap_DDA_Condition_A_Sample_Alpha_01.mzpeak"
spectra_reader = MzPeakFile(mzpeak_path)
spectrum_index = spectra_reader.spectra.reset_index().set_index("id")["index"]

def observed_spectrum(row: pd.Series) -> sus.MsmsSpectrum:
    """Fetch and annotate the observed MS2 spectrum for one PSM row."""
    raw = spectra_reader[int(spectrum_index[row["spectrum_id"]])]
    charge = int(row["peptidoform"].rsplit("/", 1)[1])
    return (
        sus.MsmsSpectrum(
            identifier=row["spectrum_id"],
            precursor_mz=row["precursor_mz"],
            precursor_charge=charge,
            mz=raw["m/z array"],
            intensity=raw["intensity array"],
        )
        .annotate_proforma(row["peptidoform"], 10, "ppm", ion_types="by")
    )

In [ ]:
example_target = df[(~df["is_decoy"]) & (df["provenance:before_rescoring_qvalue"] <= 0.01)].iloc[43]

sup.spectrum(observed_spectrum(example_target))
print(example_target["peptidoform"], example_target["provenance:before_rescoring_score"])

**Try it:** Pick a decoy PSM instead (`df[df["is_decoy"]].iloc[0]`) and plot its spectrum with `observed_spectrum(...)`. Are as many peaks annotated as for the target PSM above? What does that tell you about how much of the spectrum a wrong peptide can "explain"?

In [ ]:
# example_decoy = ...

# sup.spectrum(observed_spectrum(example_decoy))

## 2. Modelling retention time with DeepLC

[DeepLC](https://github.com/compomics/deeplc) is a deep learning model that predicts a peptide's retention time (RT), when it should elute from the LC column, purely from its sequence and modifications.

A peptide has one true, sequence-determined retention behavior. If a PSM is correct, the predicted and observed RT should agree closely. If a PSM is a chance match to the wrong peptide, there is no reason for that agreement to hold. This makes the RT prediction error a powerful, and importantly independent, piece of evidence.

### 2.1 The chromatogram, for context

Before predicting anything, let's look at what a chromatogram actually is: total ion intensity over the course of the LC gradient. Below it, a histogram of the observed retention time of every identified PSM shows where, on that same timeline, our identifications actually sit.

In [ ]:
tic = spectra_reader.read_chromatogram(0)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5], vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=tic["time array"], y=tic["intensity array"], name="TIC"), row=1, col=1)
for is_decoy, group in df.groupby("is_decoy"):
    fig.add_trace(
        go.Histogram(x=group["retention_time"], name="Decoy" if is_decoy else "Target", opacity=0.6),
        row=2, col=1,
    )
fig.update_layout(barmode="overlay", title="Chromatogram vs. observed PSM retention times")
fig.update_yaxes(title_text="Total ion current", row=1, col=1)
fig.update_yaxes(title_text="Number of PSMs", row=2, col=1)
fig.update_xaxes(title_text="Retention time (min)", row=2, col=1)
fig.show()

*The density of identified PSMs roughly follows the chromatogram's shape, but not perfectly. Where would you expect relatively more or fewer identifications: near the start and end of the gradient, or in the middle? Why? [2.1a]*

In [ ]:
fig = px.scatter(
    df, x="rescoring:predicted_retention_time_best", y="rescoring:observed_retention_time_best",
    color="is_decoy", opacity=0.1,
    category_orders={"is_decoy": [True, False]},
    color_discrete_map={False: "blue", True: "red"},
    labels={"rescoring:predicted_retention_time_best": "Predicted RT", "rescoring:observed_retention_time_best": "Observed RT"},
    title="DeepLC: predicted vs. observed retention time",
)
fig.show()

*Click "False" and "True" in the legend to toggle targets and decoys on and off separately. What is the key difference between how the two are distributed in this scatter plot? [2.2a]*

### 2.3 RT error: target vs. decoy

The scatter above already hints at the answer: for a correct PSM, predicted and observed RT should agree, so the *error* between them should be small. Let's turn that error into a single metric.

We take `-log10` of the absolute RT difference, so that a small error, a good match, becomes a large, positive number. This reshapes the distribution to mirror the search engine score plot from section 1.2: something where higher is better, for both targets and decoys.

In [ ]:
df["neg_log_rt_error"] = -np.log10(df["rescoring:rt_diff_best"].clip(lower=1e-6))

fig = px.histogram(
    df, x="neg_log_rt_error", color="is_decoy", barmode="overlay",
    labels={"neg_log_rt_error": "-log10(|predicted RT − observed RT|)", "is_decoy": "Decoy"},
    title="RT prediction error: targets vs. decoys",
)
fig.show()

*Some decoy PSMs also show a small RT error. Is this a problem for the model, or expected? Why? [2.3b]*

## 3. Modelling fragmentation with MS²PIP

[MS²PIP](https://github.com/compomics/ms2pip) predicts the relative fragment ion intensities of a peptide's MS2 spectrum: which b- and y-ions should be strong, and which should be weak or absent. Comparing this prediction to the observed spectrum gives another independent piece of evidence, this time about fragmentation rather than elution. As with DeepLC, the resulting similarity between predicted and observed fragmentation for every PSM is already part of the loaded table.

### 3.1 For fun: predict a spectrum for your own name

MS²PIP only needs a peptide sequence. It does not need to come from a real experiment. Amino acid one-letter codes cover most of the alphabet, so most names "almost" work. Replace any letter that is not a valid amino acid code (`B`, `J`, `O`, `U`, `X`, `Z` are not standard amino acids) before running this.

In [ ]:
my_name = "LENNARTMARTENS"  # edit this to a valid-amino-acid version of your own name

prediction = ms2pip.predict_single(f"{my_name}/2", model="HCD")

plt.figure()
sup.spectrum(prediction.as_spectra()[0].to_spectrum_utils())
plt.show()

### 3.2 How the spectrum changes with charge state and modifications

Fragmentation behavior is not fixed per peptide sequence. It also depends on the precursor charge state and on modifications. Mirror plots let us compare two predicted spectra directly, one on top and one flipped below.

In [ ]:
def predicted_spectrum(sequence: str, charge: int) -> sus.MsmsSpectrum:
    result = ms2pip.predict_single(f"{sequence}/{charge}", model="HCD")
    predicted, _ = result.as_spectra()
    return predicted.to_spectrum_utils()

In [ ]:
example_peptide = "SAMPLEPEPTIDEK"

fig, ax = plt.subplots(figsize=(10, 5))
sup.mirror(predicted_spectrum(example_peptide, 2), predicted_spectrum(example_peptide, 3), ax=ax)
ax.set_title(f"{example_peptide}: charge 2+ (top) vs. 3+ (bottom), both predicted")
plt.show()

**Try it:** Repeat the comparison, this time keeping the charge state fixed at 2+ but adding an oxidation on a methionine (use ProForma notation, e.g. `SAM[UNIMOD:Oxidation]PLEPEPTIDEK`). Which ions change the most?

In [ ]:
# example_peptide = ...

# fig, ax = plt.subplots(figsize=(10, 5))
# sup.mirror(predicted_spectrum(example_peptide, 2), predicted_spectrum(example_peptide, 3), ax=ax)
# ax.set_title(f"{example_peptide}: charge 2+ (top) vs. 3+ (bottom), both predicted")
# plt.show()

### 3.3 Predicted vs. observed: a good target

Now we can put the two together: fetch the real observed spectrum for one confident target PSM, predict its spectrum with MS²PIP, and compare them directly in a mirror plot.

In [ ]:
sequence, charge = example_target["peptidoform"].rsplit("/", 1)
charge = int(charge)

fig, ax = plt.subplots(figsize=(10, 5))
sup.mirror(observed_spectrum(example_target), predicted_spectrum(sequence, charge), ax=ax)
ax.set_title(f"{sequence}/{charge}: observed (top) vs. MS²PIP predicted (bottom)")
plt.show()

**Try it:** Repeat this for a decoy PSM (`example_decoy`). How well does the predicted spectrum match the observed one, compared to the target above?

### 3.4 Fragmentation similarity: target vs. decoy

In [ ]:
fig = px.histogram(
    df, x="rescoring:spec_pearson_norm", color="is_decoy", barmode="overlay",
    labels={"rescoring:spec_pearson_norm": "Predicted vs. observed spectrum correlation", "is_decoy": "Decoy"},
    title="MS²PIP fragmentation similarity: targets vs. decoys",
)
fig.show()

## 4. Combining the evidence: rescoring

Each feature above is individually informative, but imperfect. MS²Rescore already combined them into a single, more discriminative score for us, using [`ristretto`](https://github.com/CompOmics/ristretto), a lightweight Python reimplementation of the Percolator algorithm: a classifier trained on the target/decoy labels, in the same run that produced the table we have used throughout this tutorial. The original Sage score and q-value are kept alongside the rescored ones, so we can compare before and after directly.

### 4.1 Do the features agree with each other?

Before comparing old and new scores, let's look at how the two ML-derived features relate to one another, and to the target/decoy label.

*Do you see PSMs that score well on one feature (RT error) but poorly on the other (fragmentation similarity)? What might that combination mean for such a PSM's true identity? [4.1a]*

### 4.2 Old score vs. new score

The scatter below compares each PSM's original Sage score to its rescored value, with the 1% FDR threshold marked on both axes.

In [ ]:
old_threshold = df.loc[(df["provenance:before_rescoring_qvalue"] <= 0.01) & (~df["is_decoy"]), "provenance:before_rescoring_score"].min()
new_threshold = df.loc[(df["qvalue"] <= 0.01) & (~df["is_decoy"]), "score"].min()

fig = px.scatter(
    df, x="provenance:before_rescoring_score", y="score", color="is_decoy", opacity=0.3,
    labels={"provenance:before_rescoring_score": "Original Sage score", "score": "Rescored score"},
    title="Old vs. new score, with 1% FDR thresholds",
)
fig.add_vline(x=old_threshold, line_dash="dash")
fig.add_hline(y=new_threshold, line_dash="dash")
fig.show()

*Four quadrants are formed by the two threshold lines. Which quadrant contains PSMs that were rejected before but are accepted after rescoring? Which contains the opposite case? [4.2a]*

In [ ]:
df["accepted_before"] = (~df["is_decoy"]) & (df["provenance:before_rescoring_qvalue"] <= 0.01)
df["accepted_after"] = (~df["is_decoy"]) & (df["qvalue"] <= 0.01)

def classify(row):
    if row["accepted_before"] and row["accepted_after"]:
        return "retained"
    if row["accepted_before"] and not row["accepted_after"]:
        return "lost"
    if not row["accepted_before"] and row["accepted_after"]:
        return "gained"
    return "never identified"

df["status"] = df.apply(classify, axis=1)
counts = df["status"].value_counts().reindex(["lost", "retained", "gained"]).fillna(0)

fig = go.Figure()
fig.add_trace(go.Bar(y=["PSMs"], x=[-counts["lost"]], name="lost", orientation="h", marker_color="red"))
fig.add_trace(go.Bar(y=["PSMs"], x=[counts["retained"]], name="retained", orientation="h", marker_color="blue"))
fig.add_trace(go.Bar(y=["PSMs"], x=[counts["gained"]], name="gained", orientation="h", marker_color="green"))
fig.update_layout(barmode="relative", title="Effect of rescoring at 1% FDR", xaxis_title="Number of PSMs")
fig.show()

*Some PSMs that passed the 1% FDR threshold before rescoring are `lost` afterwards. These are not simply removed at random: look at their `neg_log_rt_error` and `rescoring:spec_pearson_norm` values. What do they have in common? Is losing them a good or a bad outcome for the overall dataset? [4.3a]*

**Try it:** Recompute the `gained`/`lost`/`retained` counts at 5% FDR instead of 1%. Does rescoring help more or less at a looser threshold? Why might that be?

## 5. Summary

A search engine score alone often cannot fully separate correct from incorrect PSMs. DeepLC and MS²PIP predict independent physicochemical properties (retention time, fragmentation) directly from peptide sequence, without needing the experiment itself. Comparing predictions to observations turns each property into a new, discriminative feature.

A classifier trained on the target/decoy labels combines these features into one rescored value. This typically identifies more correct PSMs at the same FDR threshold, while also removing some previously accepted matches whose additional evidence did not hold up.

This same principle, predict, compare, combine, underlies most current AI applications in proteomics identification, including MS²Rescore's full automated workflow.